# Prompt Engineering

A refresher on shaping the *input* to an LLM so the *output* is what you want — reliably, cheaply, and without touching the model weights.

**Domain:** LLM Inference, Training & Optimization · **runnable:** yes

## 1. What & Why

**Prompt engineering** is the practice of designing the text (and structure) you send to a language model to steer its behavior. It is the cheapest, fastest lever you have: no training data, no GPUs, no deployment — just words. A better prompt can turn a 60%-accurate task into a 95%-accurate one in minutes.

**The problem it solves.** A pretrained/instruction-tuned LLM is a general next-token predictor. Out of the box it doesn't know *your* task, *your* output format, or *your* edge cases. Prompt engineering encodes all of that in-context, at inference time, so the model conditions its generation on exactly the right framing.

**The core techniques, in rough order of power-per-effort:**

- **Clear instructions** — say precisely what you want, the format, and the constraints.
- **Role / system prompt** — set persona, tone, and global rules separately from the user turn.
- **Few-shot (in-context learning)** — show 1–N input→output examples so the model infers the pattern. See [[few-shot-learning]].
- **Chain-of-thought** — ask the model to reason step by step before answering. See [[chain-of-thought]].
- **Output formatting / structure** — demand JSON, a schema, delimiters, or an enum to make outputs machine-parseable.
- **Decomposition** — break a hard task into a pipeline of smaller prompts.

**When to reach for it:** *always first.* Before fine-tuning ([[qlora]], [[trl-rlhf-dpo]]), before RAG ([[rag]]), before anything expensive — try to solve the task with a better prompt. **When not to:** when you need new *knowledge* the model lacks (use RAG), new *behavior* it can't be steered into (fine-tune), or when prompts have grown so long/brittle that a tuned model is cheaper per call.

## 2. Mental Model

Think of the LLM as an extremely well-read improv actor who has just walked on stage with **no memory of any previous scene**. The prompt is the *entire* briefing: the character (system prompt), the script-so-far (conversation + examples), and the cue for the next line (your question).

The actor will commit fully to whatever role and pattern you establish. Give a vague cue and you get a plausible-but-generic improvisation. Give a precise role, two worked examples, and an explicit "respond only with valid JSON" and the actor snaps into exactly that pattern.

```
        ┌──────────────────────── PROMPT (the only context) ─────────────────────────┐
        │                                                                             │
  system│  "You are a precise classifier. Reply with one word: positive/negative."    │
        │                                                                             │
  few-  │  Input: "loved it"   -> positive                                            │
  shot  │  Input: "total waste" -> negative                                           │
        │                                                                             │
  query │  Input: "it was fine, nothing special" ->                                   │
        └──────────────────────────────┬──────────────────────────────────────────────┘
                                        ▼
                          model continues the pattern: "neutral"  (or "negative")
```

Everything is **next-token prediction conditioned on the prompt**. There is no hidden state, no "it should know what I meant." If it's not in the prompt (or the model's pretraining), it isn't there.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Zero-shot** | Instruction only, no examples. Cheapest; relies entirely on the model's pretrained ability. |
| **Few-shot** | A handful of input→output demonstrations in the prompt. The model does *in-context learning* — pattern-matching, not weight updates. |
| **System prompt** | A separate, higher-priority instruction channel that sets persona and global rules. Persists across user turns. |
| **Chain-of-thought (CoT)** | Eliciting intermediate reasoning ("think step by step") before the final answer. Trades tokens for accuracy on multi-step tasks. |
| **Delimiters** | Explicit markers (```triple backticks```, `<tags>`, `###`) that separate instructions from data — critical to avoid the data being read as instructions. |
| **Output schema** | A demanded structure (JSON, enum, key:value) that makes the response parseable and constrains the model. |
| **Temperature** | Sampling randomness. ~0 for deterministic/extraction tasks, higher (0.7–1.0) for creative variety. |
| **Prompt injection** | An attack where untrusted input contains instructions that hijack the model. The reason you delimit and never trust interpolated data. |
| **Context window** | The token budget for prompt + output. Long prompts (many few-shot examples, big documents) cost money and can crowd out the answer. |
| **Idempotence of phrasing** | Small wording changes can swing outputs. Prompts are code: version them, test them, measure them. |

## 4. Setup

Prompt engineering itself needs **no libraries** — it's just strings. The runnable examples below use only the Python standard library so they execute in any fresh kernel.

The final example calls a real LLM (Anthropic's API) but is **gated behind an `os.getenv` check**, so the notebook runs top-to-bottom whether or not you have a key. To run that cell for real:

```bash
%pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import os
import re
import json
import textwrap

print("stdlib only — no install needed for the local examples")
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## 5. Worked Examples

### Example 1 — Zero-shot vs. few-shot, and why structure matters

We'll build prompts with a small helper, then run them against a tiny **deterministic stand-in model** so the notebook is reproducible offline. The stand-in is a keyword classifier — it is *not* a real LLM, but it lets us demonstrate prompt *construction* and *output parsing* without a network call. The same prompt strings are exactly what you'd send to a real model.

In [ ]:
def build_prompt(query, examples=None, system=None, output_instruction=None):
    """Assemble a prompt from parts. Stable/global content first, the query last."""
    parts = []
    if system:
        parts.append(f"[SYSTEM]\n{system}")
    if examples:
        shots = "\n".join(f"Input: {x!r} -> {y}" for x, y in examples)
        parts.append(f"[EXAMPLES]\n{shots}")
    if output_instruction:
        parts.append(f"[OUTPUT]\n{output_instruction}")
    # Delimit the untrusted user data so it can't be read as an instruction.
    parts.append(f"[QUERY]\nInput: ```{query}```\n->")
    return "\n\n".join(parts)


SYSTEM = "You are a sentiment classifier."
OUTPUT = "Reply with exactly one word: positive, negative, or neutral."
SHOTS = [("loved every minute", "positive"),
         ("total waste of money", "negative"),
         ("it was fine, nothing special", "neutral")]

zero_shot = build_prompt("the plot dragged but the acting saved it",
                         system=SYSTEM, output_instruction=OUTPUT)
few_shot = build_prompt("the plot dragged but the acting saved it",
                        system=SYSTEM, examples=SHOTS, output_instruction=OUTPUT)

print("=== ZERO-SHOT PROMPT ===")
print(zero_shot)
print("\n=== FEW-SHOT PROMPT (same query, + 3 demonstrations) ===")
print(few_shot)

Notice the few-shot prompt teaches the model the **label set and the exact output shape** by example, not just by description. Now a deterministic stand-in "model" so we can see parsing in action without a network:

In [ ]:
POS = {"loved", "great", "saved", "excellent", "good", "amazing"}
NEG = {"waste", "dragged", "bad", "terrible", "boring", "awful"}

def stand_in_model(prompt):
    """Toy deterministic 'model': scores the delimited query by keyword hits.
    Real models would condition on the [EXAMPLES]/[OUTPUT] sections too."""
    m = re.search(r"\[QUERY\].*?```(.*?)```", prompt, re.S)
    text = (m.group(1) if m else prompt).lower()
    words = set(re.findall(r"[a-z']+", text))
    pos, neg = len(words & POS), len(words & NEG)
    if pos > neg: return "positive"
    if neg > pos: return "negative"
    return "neutral"

raw = stand_in_model(few_shot)

# Robustly parse: constrain to the allowed label set rather than trusting raw text.
ALLOWED = {"positive", "negative", "neutral"}
label = next((w for w in re.findall(r"[a-z]+", raw.lower()) if w in ALLOWED), "neutral")
print("raw output:", repr(raw))
print("parsed label:", label)
assert label in ALLOWED

### Example 2 — Demanding (and safely parsing) structured JSON output

The single highest-leverage move for production prompts: make the model emit **machine-readable structure**, then parse defensively. Models often wrap JSON in prose or code fences, so never `json.loads()` the raw response blindly.

In [ ]:
EXTRACTION_PROMPT = textwrap.dedent("""\
    [SYSTEM]
    Extract contact details. Respond with ONLY a JSON object, no prose, with keys:
    name (string), email (string|null), wants_demo (boolean).

    [QUERY]
    Message: ```Hi, this is Dana Lee (dana@acme.io). Can we book a demo next week?```
    ->""")

print(EXTRACTION_PROMPT)
print("-" * 60)

# Simulate a typical messy model response: JSON wrapped in a code fence + prose.
model_response = textwrap.dedent("""\
    Sure! Here is the extracted data:
    ```json
    {"name": "Dana Lee", "email": "dana@acme.io", "wants_demo": true}
    ```
    Let me know if you need anything else.""")

def extract_json(text):
    """Pull the first {...} block out of a possibly-chatty response."""
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S)
    candidate = fenced.group(1) if fenced else re.search(r"\{.*\}", text, re.S).group(0)
    return json.loads(candidate)

data = extract_json(model_response)
print("parsed:", data)
print("wants_demo is a real bool:", data["wants_demo"] is True)
assert data["name"] == "Dana Lee" and data["wants_demo"] is True

### Example 3 — The real thing (gated): a system prompt + few-shot call to Claude

This is the production shape: a `system` prompt for global rules and `messages` carrying few-shot turns plus the query. It runs **only if `ANTHROPIC_API_KEY` is set**, so the notebook still executes cleanly without one. The call shape is shown either way.

In [ ]:
def classify_with_claude(text):
    import anthropic  # imported lazily so the notebook runs without the package
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    resp = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=16,  # we only want one word back
        system="You are a sentiment classifier. "
               "Reply with exactly one word: positive, negative, or neutral.",
        messages=[
            {"role": "user", "content": "loved every minute"},
            {"role": "assistant", "content": "positive"},
            {"role": "user", "content": "total waste of money"},
            {"role": "assistant", "content": "negative"},
            {"role": "user", "content": text},
        ],
    )
    return resp.content[0].text.strip().lower()


SAMPLE = "the plot dragged but the acting saved it"
if os.getenv("ANTHROPIC_API_KEY"):
    print("Claude says:", classify_with_claude(SAMPLE))
else:
    print("ANTHROPIC_API_KEY not set — skipping the live call.")
    print("Would send model=claude-opus-4-8 with a system prompt + 2 few-shot turns,")
    print(f"then the query: {SAMPLE!r}")

## 6. Gotchas & Pitfalls

- **Prompt injection.** If you interpolate user/web/document text into a prompt, that text can contain "ignore previous instructions" and hijack the model. **Always delimit untrusted data** (backticks, tags) and, for higher stakes, keep operator instructions in the `system` channel — it carries more authority than user-turn text.
- **Trusting raw output.** Models wrap JSON in prose or code fences, add trailing commentary, or emit a synonym of your label. Parse defensively (extract-then-`json.loads`, constrain to an allowed set) — see Examples 1 and 2.
- **Over-stuffing few-shot examples.** More shots cost tokens and latency and can *bias* toward the example distribution. 2–5 well-chosen, diverse examples usually beat 20 redundant ones.
- **Aggressive "CRITICAL: YOU MUST" phrasing backfires on modern models.** Recent instruction-tuned models follow plain instructions closely; shouty over-steering causes *over*-triggering (e.g. calling a tool every turn). Say it once, plainly.
- **Asking for reasoning *and* a clean parse in one shot.** Chain-of-thought emits lots of text; if you also need strict JSON, either use two steps (reason, then format) or a structured-output feature — don't regex a label out of a paragraph of reasoning.
- **Temperature mismatch.** Leaving `temperature` high on an extraction/classification task adds needless nondeterminism; drop it to ~0. (Note: some newer Claude models remove `temperature` entirely and steer via prompting instead.)
- **Silent context-window overflow.** Huge documents + many examples can push the actual answer past the limit or truncate it. Budget tokens; count them ([[kv-cache]] explains why long prompts also cost latency).
- **Treating prompts as throwaway.** Wording changes swing accuracy. Version prompts, keep a small eval set, and measure changes — prompts are code.

## 7. When to Use vs Alternatives

| Approach | Best when | Cost / downside |
|---|---|---|
| **Prompt engineering** | Always try first. Task is expressible in instructions + a few examples; you need flexibility and zero training. | Limited by the model's pretrained knowledge/skill; long prompts cost tokens every call; can get brittle. |
| **Few-shot in-context learning** | A handful of examples nails the pattern/format. | Examples eat context window on every request. |
| **RAG** ([[rag]], [[semantic-search]]) | The model lacks *knowledge* (private docs, fresh facts). | Adds a retrieval system; prompt still needed to use the retrieved context well. |
| **Fine-tuning** ([[qlora]], [[trl-rlhf-dpo]]) | A fixed behavior/format is needed at scale, or prompts have grown huge and brittle; you want to bake the pattern into the weights to shrink per-call tokens. | Needs labeled data, training compute, and a deploy/serving path; less flexible to change. |
| **Tool use / function calling** | The task needs real actions or precise computation (search, code, DB). | Orchestration complexity; still relies on good prompting for tool selection. |

**Rule of thumb:** prompt → few-shot → RAG (for knowledge) → fine-tune (for behavior at scale). They compose: a fine-tuned model still benefits from a good prompt, and RAG is just prompt engineering with retrieved context. Reach down the list only when the cheaper rung above genuinely can't do the job.

## 8. Resources

- **Anthropic — Prompt engineering overview**: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview
- **OpenAI — Prompt engineering guide**: https://platform.openai.com/docs/guides/prompt-engineering
- **Google — Prompt Engineering whitepaper (Kaggle, Lee Boonstra, 2024)**: https://www.kaggle.com/whitepaper-prompt-engineering
- **DAIR.ai — Prompt Engineering Guide** (broad, technique-by-technique): https://www.promptingguide.ai/
- **Brown et al., 2020 — "Language Models are Few-Shot Learners" (GPT-3)**: https://arxiv.org/abs/2005.14165
- **Wei et al., 2022 — "Chain-of-Thought Prompting"**: https://arxiv.org/abs/2201.11903
- Related notebooks in this domain: [[few-shot-learning]], [[chain-of-thought]], [[rag]].